# Build Vector Store — OPD Kab Batang

Notebook ini dijalankan **sekali** untuk:
1. Load `cleaned_opd_docs.pkl` hasil preprocessing OPD
2. Embed semua 61 records langsung (tanpa chunking — tiap record sudah atomic)
3. Persist ke ChromaDB di `data/opd_vector_store/` (collection `opd_directory`)

Output: vectorstore siap dipakai `rag_chat_opd.ipynb`.

## Step 1 — Load cleaned OPD docs

In [1]:
import pickle
from pathlib import Path
from collections import Counter

with open("../data/cleaned_opd_docs.pkl", "rb") as f:
    cleaned_opd_docs = pickle.load(f)

print(f"Loaded {len(cleaned_opd_docs)} OPD docs")
print("\nDistribusi tipe:")
for tipe, count in Counter(d.metadata['tipe'] for d in cleaned_opd_docs).most_common():
    print(f"  {tipe:15s}: {count}")

print(f"\nSample doc:")
print(cleaned_opd_docs[0].page_content)
print(f"Metadata: {cleaned_opd_docs[0].metadata}")

Loaded 61 OPD docs

Distribusi tipe:
  Dinas          : 17
  Kecamatan      : 15
  Bagian         : 9
  Kelurahan      : 9
  Badan          : 4
  Sekretariat    : 2
  RSUD           : 2
  Inspektorat    : 1
  Satpol         : 1
  Kantor         : 1

Sample doc:
Nama OPD: Sekretariat Daerah
Tipe: Sekretariat
Alamat: Jl. RA Kartini No. 1 Batang
Email: -
No. Telp: (0285) 391571
Metadata: {'source': 'Nama dan Alamat OPD Kab Batang.pdf', 'doc_type': 'opd_directory', 'nomor': '1', 'nama_opd': 'Sekretariat Daerah', 'parent_opd': '', 'tipe': 'Sekretariat', 'page': 1, 'has_email': False, 'has_telp': True}


## Step 2 — Embedding pakai Gemini

Pakai model & dimensi yang sama dengan `build_vectorstore.ipynb` untuk konsistensi.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert api_key, "GEMINI_API_KEY tidak ditemukan di .env"
os.environ["GOOGLE_API_KEY"] = api_key
print(f"API key loaded ({len(api_key)} chars)")

API key loaded (39 chars)


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_document",
    output_dimensionality=768,
)

sample_vec = embeddings.embed_query("test")
print(f"Embedding dim: {len(sample_vec)}")

c:\Users\Nafisha\Documents\RAGTrial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Embedding dim: 768


## Step 3 — Build & Persist ChromaDB

Hanya 61 docs → 1 batch cukup (jauh di bawah rate limit 100/menit). Wipe path lama dulu.

In [4]:
import shutil

VS_PATH = Path("../data/opd_vector_store")
if VS_PATH.exists():
    shutil.rmtree(VS_PATH)
    print(f"Wiped {VS_PATH.resolve()}")
VS_PATH.mkdir(parents=True, exist_ok=True)

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=cleaned_opd_docs,
    embedding=embeddings,
    collection_name="opd_directory",
    persist_directory=str(VS_PATH),
)

count = vectorstore._collection.count()
print(f"✓ Vector store ready: {count} chunks ter-embed & ter-persist")
assert count == len(cleaned_opd_docs), f"Mismatch: {count} vs {len(cleaned_opd_docs)}"

✓ Vector store ready: 61 chunks ter-embed & ter-persist


## Step 4 — Sanity Check Retrieval

In [6]:
test_queries = [
    "alamat dinas pariwisata",
    "nomor telp kecamatan batang",
    "email bagian kesra",
]

for q in test_queries:
    print(f"\n>>> Query: {q}")
    results = vectorstore.similarity_search(q, k=3)
    for i, r in enumerate(results, 1):
        m = r.metadata
        print(f"  [{i}] [{m['nomor']}] {m['nama_opd']} ({m['tipe']})")
        preview = r.page_content.replace('\n', ' | ')
        print(f"      {preview}")


>>> Query: alamat dinas pariwisata
  [1] [12] Dinas Pariwisata, Kepemudaan dan Olahraga (Dinas)
      Nama OPD: Dinas Pariwisata, Kepemudaan dan Olahraga | Tipe: Dinas | Alamat: Jl. Urip Sumoharjo No. 34 Batang | Email: disparpora@batangkab.go.id | No. Telp: (0285) 391141
  [2] [13] Dinas Penanaman Modal, Pelayanan
Terpadu Satu Pintu dan Tenaga Kerja (Dinas)
      Nama OPD: Dinas Penanaman Modal, Pelayanan | Terpadu Satu Pintu dan Tenaga Kerja | Tipe: Dinas | Alamat: Jl. Urip Sumoharjo No. 13 Batang | Email: dpmptspnaker@batangkab.go.id | No. Telp: (0285) 4493081
  [3] [7] Dinas Pendidikan dan Kebudayaan (Dinas)
      Nama OPD: Dinas Pendidikan dan Kebudayaan | Tipe: Dinas | Alamat: Jl. Slamet Riyadi No. 29 Batang | Email: disdikbud@batangkab.go.id | No. Telp: (0285) 391321

>>> Query: nomor telp kecamatan batang
  [1] [25] Kecamatan Batang (Kecamatan)
      Nama OPD: Kecamatan Batang | Tipe: Kecamatan | Alamat: Jl. Perintis Kemerdekaan No. 01 Batang | Email: kec_batang@batangkab.go.i

## Done

Vector store ter-persist di `data/opd_vector_store/`. Lanjut ke `rag_chat_opd.ipynb` untuk eksperimen retrieval & generation.